# Kaggle Data Pipeline Benchmark & Verification (CPU)

This notebook rigorously tests the upgraded `src/` data pipeline on the full dataset in a Kaggle environment (CPU-only).
It verifies dataset layout, bounds preprocessing multithreading, builds the memmap cache for the ENTIRE dataset, and benchmarks the PyTorch data loader for precise throughput and time telemetry.

In [ ]:
import os
import sys
import time
import pandas as pd

# 1. Attach Repository Root
REPO_ROOT = '/kaggle/working/Knee_RSNA'
if os.path.exists(REPO_ROOT) and REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
elif os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

from src import config
from src.preprocess import runner
from src.preprocess.cache import cfg_of
from src.dataset import RSNADataset
from torch.utils.data import DataLoader

print('Pipeline imports successful!')

In [ ]:
# 2. Verify Dataset Location & Layout
KAGGLE_DATA_ROOT = '/kaggle/input/rsna-knee-abnormality-detection'
print(f'Checking dataset root: {KAGGLE_DATA_ROOT}')
if not os.path.exists(KAGGLE_DATA_ROOT):
    print('WARNING: Kaggle dataset root not found. Please attach the RSNA competition dataset.')
else:
    print('Dataset attached. Found the following directories:')
    print(os.listdir(KAGGLE_DATA_ROOT))

In [ ]:
# 3. Run Preprocessing Indexer (FULL DATASET)
print('\n--- Building/Verifying Series Index (Full Dataset) ---')
cfg = config.get_cfg('v2')

try:
    t0 = time.time()
    # limit=0 means process the entire dataset (train & test)
    index_out = runner.get_index(KAGGLE_DATA_ROOT, '/kaggle/working/idx', splits=('train', 'test'), limit=0, force=True)
    index_df = index_out['ann']
    total_time = time.time() - t0
    print(f'\n[TELEMETRY] Index built successfully in {total_time:.2f}s!')
    print(f'[TELEMETRY] Total series found: {len(index_df)}')
    print(f'[TELEMETRY] Throughput: {len(index_df) / max(total_time, 1e-6):.2f} series/sec')
except Exception as e:
    print(f'Index building failed: {e}')

In [ ]:
# 4. Run Cache Generation (FULL DATASET STRESS TEST)
print('\n--- Testing Cache Generation (CPU/RAM Multi-Core Stress) ---')
CACHE_DIR = '/kaggle/working/test_cache'
os.makedirs(os.path.dirname(CACHE_DIR), exist_ok=True)

try:
    # Automatically detect available cores for max load
    cpu_cores = max(2, os.cpu_count() or 2)
    print(f'Detected {cpu_cores} CPU cores. Launching multiprocessing workers...')
    
    # Test cache generation on the full training dataset
    train_studies = index_df[index_df['split'] == 'train']['StudyInstanceUID'].unique()
    
    t0 = time.time()
    cache, stats = runner.run_cache(
        index_out, 
        'train', 
        CACHE_DIR, 
        cfg=cfg, 
        workers=cpu_cores, 
        studies=train_studies, 
        fresh=True
    )
    cache_time = time.time() - t0
    print(f'\n[TELEMETRY] Full cache built in {cache_time:.2f}s!')
    print(f'[TELEMETRY] Total studies processed: {len(train_studies)}')
    print(f'[TELEMETRY] Cache Build Throughput: {len(train_studies) / max(cache_time, 1e-6):.2f} studies/sec')
    print('\nDetailed Cache Stats:', stats)
    cache.flush()
except Exception as e:
    print(f'Cache generation failed: {e}')

In [ ]:
# 5. PyTorch DataLoader Benchmarking (FULL EPOCH THROUGHPUT)
print('\n--- Testing PyTorch DataLoader Throughput ---')
train_csv = os.path.join(KAGGLE_DATA_ROOT, 'train.csv')
if os.path.exists(train_csv):
    df_train = pd.read_csv(train_csv)
    
    # Initialize PyTorch dataset mapped directly against the memmap cache
    dataset = RSNADataset(df_train, CACHE_DIR, cfg, is_train=True, n_windows_use=4, aug=False)
    
    # Kaggle multi-processing on data loaders is extremely sensitive to RAM. 
    # We benchmark single-process throughput here for absolute stability.
    batch_size = 8
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    t0 = time.time()
    total_samples = 0
    print('Starting full epoch data iteration...')
    
    for batch_idx, (imgs, slot_mask, win_mask, targets, weights) in enumerate(loader):
        total_samples += imgs.shape[0]
        
        if batch_idx % 100 == 0:
            elapsed = time.time() - t0
            print(f'Batch {batch_idx:04d} | Samples processed: {total_samples:05d} | Current Throughput: {total_samples / max(elapsed, 1e-6):.2f} samples/sec')
            
    epoch_time = time.time() - t0
    print(f'\n[TELEMETRY] DataLoader iteration completed in {epoch_time:.2f}s!')
    print(f'[TELEMETRY] Total samples iterated: {total_samples}')
    print(f'[TELEMETRY] Overall Dataloader Throughput: {total_samples / max(epoch_time, 1e-6):.2f} samples/sec')
else:
    print('train.csv not found. Are you running on Kaggle with the dataset attached?')